# Benchmarking-System - Filialvergleich
## Gruppe 18: Marco, Harun, Duy
### Phase 4 - Systemkonzept Benchmarking

Dieses Notebook ermöglicht die interaktive Analyse der Benchmark-Daten für die Filialen Rosenheim und Freiburg/Karlsruhe.

In [ ]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from db_connect import get_connection
import warnings
warnings.filterwarnings('ignore')

# Plotting-Einstellungen
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Pandas Display-Optionen
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

## 1. Datenbankverbindung und Daten laden

In [ ]:
# Verbindung herstellen
conn = get_connection()
print("Datenbankverbindung hergestellt!")

In [ ]:
# KPI-Daten laden (Fallback auf Basistabellen falls Views nicht existieren)
try:
    df_kpi = pd.read_sql("SELECT * FROM V_BENCHMARK_KPI", conn)
    print("Daten aus V_BENCHMARK_KPI geladen")
except:
    print("View nicht gefunden - lade Daten direkt aus Basistabellen...")
    query = """
    SELECT
        s.ID_STORE,
        so.STORE_LOCATION AS STORE_NAME,
        s.ID_CALMONTH,
        FORMAT(s.ID_CALMONTH, 'yyyy-MM') AS ID_CALMONTH_STD,
        pc.CATEGORY AS BENCHMARK_CATEGORY,
        SUM(s.SALES_AMOUNT) AS TOTAL_SALES_AMOUNT,
        SUM(s.REVENUE) AS REVENUE_EUR,
        SUM(s.GROSS_PROFIT_EUR) AS GROSS_PROFIT_EUR,
        AVG(s.SALES_PRICE_EUR) AS AVG_SALES_PRICE_EUR,
        SUM(s.DISCOUNT_IN_EUR) AS DISCOUNT_EUR
    FROM T_ETL_MONTHLY_SALES s
    INNER JOIN T_SALESORG so ON s.ID_STORE = so.SALESORG_ID
    INNER JOIN T_PRODUCT_CATEGORY pc ON s.ID_CATEGORY = pc.ID_CATEGORY
    WHERE so.STORE_LOCATION IN ('Rosenheim', 'Freiburg', 'Karlsruhe')
    GROUP BY s.ID_STORE, so.STORE_LOCATION, s.ID_CALMONTH, pc.CATEGORY
    ORDER BY s.ID_CALMONTH, so.STORE_LOCATION
    """
    df_kpi = pd.read_sql(query, conn)
    # KPIs berechnen
    df_kpi['GROSS_MARGIN_PCT'] = (df_kpi['GROSS_PROFIT_EUR'] / df_kpi['REVENUE_EUR'] * 100).round(2)

print(f"\nAnzahl Datensätze: {len(df_kpi)}")
df_kpi.head(10)

## 2. Übersicht der Daten

In [ ]:
# Verfügbare Filialen
print("Verfügbare Filialen:")
for store in df_kpi['STORE_NAME'].unique():
    count = len(df_kpi[df_kpi['STORE_NAME'] == store])
    print(f"  - {store}: {count} Datensätze")

print(f"\nZeitraum: {df_kpi['ID_CALMONTH'].min()} bis {df_kpi['ID_CALMONTH'].max()}")

print("\nVerfügbare Kategorien:")
for cat in df_kpi['BENCHMARK_CATEGORY'].unique():
    print(f"  - {cat}")

## 3. Umsatzanalyse

In [ ]:
# Gesamtumsatz pro Filiale
df_revenue = df_kpi.groupby('STORE_NAME').agg({
    'REVENUE_EUR': 'sum',
    'GROSS_PROFIT_EUR': 'sum',
    'TOTAL_SALES_AMOUNT': 'sum'
}).round(2)

df_revenue.columns = ['Umsatz (EUR)', 'Bruttogewinn (EUR)', 'Verkaufte Einheiten']
print("Gesamtübersicht pro Filiale:")
df_revenue

In [ ]:
# Umsatz-Trend Chart
df_monthly = df_kpi.groupby(['ID_CALMONTH', 'STORE_NAME'])['REVENUE_EUR'].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 6))

for store in df_monthly['STORE_NAME'].unique():
    store_data = df_monthly[df_monthly['STORE_NAME'] == store]
    ax.plot(store_data['ID_CALMONTH'], store_data['REVENUE_EUR'] / 1000,
            marker='o', linewidth=2, markersize=6, label=store)

ax.set_xlabel('Monat', fontsize=12)
ax.set_ylabel('Umsatz (Tsd. EUR)', fontsize=12)
ax.set_title('Monatlicher Umsatz nach Filiale', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Margenanalyse

In [ ]:
# Durchschnittliche Margen pro Filiale
df_margins = df_kpi.groupby('STORE_NAME').agg({
    'GROSS_MARGIN_PCT': 'mean'
}).round(2)

print("Durchschnittliche Bruttomargen:")
df_margins

In [ ]:
# Margen-Balkendiagramm
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#3498db', '#e74c3c', '#2ecc71']
bars = ax.bar(df_margins.index, df_margins['GROSS_MARGIN_PCT'], color=colors[:len(df_margins)])

ax.set_xlabel('Filiale', fontsize=12)
ax.set_ylabel('Bruttomarge (%)', fontsize=12)
ax.set_title('Durchschnittliche Bruttomarge nach Filiale', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Werte auf Balken
for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height),
               ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Kategorieanalyse

In [ ]:
# Umsatz nach Kategorie und Filiale
df_cat = df_kpi.groupby(['BENCHMARK_CATEGORY', 'STORE_NAME'])['REVENUE_EUR'].sum().unstack().fillna(0)

print("Umsatz nach Kategorie (EUR):")
df_cat

In [ ]:
# Kategorien-Balkendiagramm
fig, ax = plt.subplots(figsize=(14, 7))

df_cat.plot(kind='bar', ax=ax, width=0.8, colormap='Set2')

ax.set_xlabel('Fahrradkategorie', fontsize=12)
ax.set_ylabel('Umsatz (EUR)', fontsize=12)
ax.set_title('Umsatz nach Kategorie und Filiale', fontsize=14, fontweight='bold')
ax.legend(title='Filiale', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45, ha='right')

# Y-Achse formatieren
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1000:.0f}k'))

plt.tight_layout()
plt.show()

## 6. Filialvergleich (Benchmark)

In [ ]:
# Vergleich der Filialen
try:
    df_comparison = pd.read_sql("SELECT * FROM V_BENCHMARK_STORE_COMPARISON", conn)
    print("Daten aus V_BENCHMARK_STORE_COMPARISON geladen")
    df_comparison.head()
except Exception as e:
    print(f"View nicht verfügbar: {e}")
    print("Erstelle manuelle Vergleichsanalyse...")
    
    # Manuelle Aggregation
    df_store_comparison = df_kpi.groupby(['ID_CALMONTH_STD', 'STORE_NAME']).agg({
        'REVENUE_EUR': 'sum',
        'GROSS_PROFIT_EUR': 'sum',
        'GROSS_MARGIN_PCT': 'mean'
    }).round(2)
    
    df_store_comparison

In [ ]:
# Pivot-Vergleich
df_pivot = df_kpi.groupby(['ID_CALMONTH_STD', 'STORE_NAME'])['REVENUE_EUR'].sum().unstack().fillna(0)

# Differenz berechnen falls beide Filialen vorhanden
if 'Rosenheim' in df_pivot.columns:
    ros_col = 'Rosenheim'
    other_cols = [c for c in df_pivot.columns if c != 'Rosenheim']
    if other_cols:
        df_pivot['Differenz'] = df_pivot[ros_col] - df_pivot[other_cols[0]]

print("Monatlicher Umsatzvergleich:")
df_pivot

In [ ]:
# Vergleichs-Chart
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Oberes Chart: Umsatz pro Filiale
x = range(len(df_pivot))
width = 0.35

stores = [c for c in df_pivot.columns if c != 'Differenz']
colors = ['#3498db', '#e74c3c']

for i, store in enumerate(stores[:2]):
    offset = (i - 0.5) * width
    ax1.bar([xi + offset for xi in x], df_pivot[store] / 1000, width, label=store, color=colors[i])

ax1.set_xlabel('Monat')
ax1.set_ylabel('Umsatz (Tsd. EUR)')
ax1.set_title('Monatlicher Umsatzvergleich', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(df_pivot.index, rotation=45)
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# Unteres Chart: Differenz (falls vorhanden)
if 'Differenz' in df_pivot.columns:
    colors_diff = ['#2ecc71' if x >= 0 else '#e74c3c' for x in df_pivot['Differenz']]
    ax2.bar(x, df_pivot['Differenz'] / 1000, color=colors_diff)
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax2.set_xlabel('Monat')
    ax2.set_ylabel('Differenz (Tsd. EUR)')
    ax2.set_title(f'Umsatzdifferenz (Rosenheim - {stores[1] if len(stores) > 1 else "Andere"})', fontsize=14, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(df_pivot.index, rotation=45)
    ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 7. KPI-Zusammenfassung

In [ ]:
# KPI-Zusammenfassung erstellen
kpi_summary = df_kpi.groupby('STORE_NAME').agg({
    'REVENUE_EUR': ['sum', 'mean'],
    'GROSS_PROFIT_EUR': ['sum', 'mean'],
    'TOTAL_SALES_AMOUNT': 'sum',
    'GROSS_MARGIN_PCT': 'mean'
}).round(2)

kpi_summary.columns = ['Gesamtumsatz', 'Ø Monatsumsatz', 'Gesamtgewinn', 'Ø Monatsgewinn', 
                       'Verkaufte Einheiten', 'Ø Bruttomarge %']

print("\n" + "="*60)
print("KPI-ZUSAMMENFASSUNG")
print("="*60)
kpi_summary

## 8. Export nach Excel

In [ ]:
# Excel-Export
with pd.ExcelWriter('benchmark_analysis_export.xlsx', engine='openpyxl') as writer:
    # Rohdaten
    df_kpi.to_excel(writer, sheet_name='KPI_Daten', index=False)
    
    # Monatliche Übersicht
    df_monthly_export = df_kpi.groupby(['ID_CALMONTH_STD', 'STORE_NAME']).agg({
        'REVENUE_EUR': 'sum',
        'GROSS_PROFIT_EUR': 'sum',
        'TOTAL_SALES_AMOUNT': 'sum',
        'GROSS_MARGIN_PCT': 'mean'
    }).round(2).reset_index()
    df_monthly_export.to_excel(writer, sheet_name='Monatsuebersicht', index=False)
    
    # KPI-Zusammenfassung
    kpi_summary.to_excel(writer, sheet_name='KPI_Summary')
    
    # Kategorien
    df_cat.to_excel(writer, sheet_name='Nach_Kategorie')

print("Excel-Export erstellt: benchmark_analysis_export.xlsx")

In [ ]:
# Verbindung schließen
conn.close()
print("Datenbankverbindung geschlossen.")